In [1]:
import os 
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

In [2]:
# Load data
model_input = pd.read_parquet("/data/big/fmoss/data/model_input_dataset/training_dataset.parquet")
# Add year column based on DisNo.
model_input["year"] = model_input["DisNo."].apply(lambda x: int(x[:4]))

# Filtering
# df = model_input.copy()
# # People affected > 100
# events_to_consider = df[df["Total Affected"] > 100]["DisNo."].unique()
# df = df[df["DisNo."].isin(events_to_consider)]
# # Only subnational reported events
# df = df[df.level != "ADM0"]

df = model_input.copy()

# People affected > 100
events_to_consider_c1 = df[df["Total Affected"] > 100]["DisNo."].unique()
events_to_consider_c2 = df[df["level"] != "ADM0"]["DisNo."].unique()

# More constraints regarding subnational reported events
proportions = (
    df[df["level"] != "ADM0"]
    .assign(is_affected=lambda x: x["Total Affected"] > 0)
    .groupby(["DisNo.", "GID_0", "GID_1"])["is_affected"].max()
    .groupby(["DisNo.", "GID_0"]).mean()
    .reset_index(name="proportion_affected_gid1")
)

events_to_consider_c3 = proportions[proportions.proportion_affected_gid1 < 1]["DisNo."].unique()

events_to_consider = list(set(events_to_consider_c1) & set(events_to_consider_c2) & set(events_to_consider_c3))
print(len(events_to_consider))

df = df[df["DisNo."].isin(events_to_consider)]


780


In [3]:
# missing events for the xgb model
# missing_events_1 = [
#     '2003-0346-CHN',
#     '2003-0346-HKG',
#     '2003-0443-CHN',
#     '2006-0410-PHL',
#     '2006-0415-PHL',
#     '2007-0457-PHL',
#     '2007-0668-PHL',
#     '2008-0329-CHN',
#     '2008-0369-CHN',
#     '2008-0369-HKG',
#     '2009-0399-CHN',
#     '2009-0399-HKG',
#     '2017-0352-CHN',
#     '2017-0352-HKG'
#     ]

missing_events = [
    '2009-0399-CHN',
    '2003-0346-CHN',
    '2003-0346-HKG',
    '2017-0352-HKG',
    '2009-0399-HKG',
    '2008-0369-CHN',
    '2007-0457-PHL',
    '2006-0410-PHL',
    '2017-0352-CHN',
    '2008-0369-HKG',
    '2006-0415-PHL',
    '2007-0668-PHL'
    ]
df = df[~df["DisNo."].isin(missing_events)]


In [4]:
def compute_historical_predictions(df, threshold=33):
    # Precompute for each country + year + event the median of previous % impacts
    # First build a mapping of country -> list of (year, DisNo., median)
    country_events = (
        df.groupby(["GID_0", "DisNo.", "year"])
        ["perc_affected_pop_grid_region"]
        .apply(lambda x: x.unique())
        .reset_index()
    )

    # Remove zeros in the unique percents
    country_events["impact_percent"] = country_events["perc_affected_pop_grid_region"].apply(
        lambda arr: [v for v in arr if v != 0]
    )

    # We don't need the original array anymore
    country_events.drop(columns="perc_affected_pop_grid_region", inplace=True)

    # Build lookup table: for each (GID_0, year, DisNo.), the median of past events
    result_rows = []
    for _, row in country_events.iterrows():
        gid0 = row["GID_0"]
        disno = row["DisNo."]
        year = row["year"]

        # Get all past events for this country with year <= current year, and different event
        past = country_events[
            (country_events["GID_0"] == gid0)
            & (country_events["year"] <= year)
            & (country_events["DisNo."] != disno)
        ]

        # Collect all impact percentages from past
        past_impacts = [val for sublist in past["impact_percent"] for val in sublist]

        if past_impacts:
            median_pct = np.median(past_impacts)
        else:
            median_pct = 0

        result_rows.append((gid0, disno, median_pct))

    # Build a dataframe for quick merging
    median_df = pd.DataFrame(result_rows, columns=["GID_0", "DisNo.", "median_pct"])

    # Merge back to original df
    df = df.merge(median_df, on=["GID_0", "DisNo."], how="left")

    # Compute predictions
    df["predicted_flag"] = (df["wind_speed"] > threshold).astype(int)
    df["predicted"] = (df["median_pct"] / 100.0) * df["population"] * df["predicted_flag"]
    df["reported_ppl"] = df["population"] * df["perc_affected_pop_grid_region"] / 100
    df["reported"] = df["perc_affected_pop_grid_region"]

    # Final selection
    df_predictions = df[
        ["DisNo.", "GID_0", "GID_1", "id", "level", "population",
         "wind_speed", "Total Affected", "predicted", "reported", "reported_ppl"]
    ].copy()

    return df_predictions


In [15]:
df.columns

Index(['DisNo.', 'GID_0', 'sid', 'id', 'level', 'Total Affected',
       'N_events_5_years', 'wind_speed', 'rainfall_max_24h', 'track_distance',
       'perc_affected_pop_grid_region', 'GID_1', 'GID_2', 'iso3', 'population',
       'coast_length', 'with_coast', 'mean_elev', 'mean_slope', 'mean_rug',
       'urban', 'rural', 'water', 'shdi', 'storm_tide_rp_0010',
       'landslide_risk_sum', 'flood_risk', 'year'],
      dtype='object')

In [5]:
# def compute_grouped_predictions(df_predictions, level="ADM0"):
#     if level == "ADM0":
#         group_keys = ["DisNo.", "GID_0"]
#         grouped = (
#             df_predictions.groupby(group_keys)
#             .agg(
#                 prediction_ppl=("predicted", "sum"),
#                 actual_ppl=("reported_ppl", "sum"),
#                 # population=("population", lambda x: x[df_predictions.loc[x.index, "Total Affected"] > 0].sum()),
#                 population=("population", "sum"),
#                 total_affected=("Total Affected", "max"),
#                 wind_speed=("wind_speed", "max"),
#             )
#             .reset_index()
#         )
#         grouped["reported"] = 100 * grouped["total_affected"] / grouped["population"]
#         grouped["reported"] = grouped["reported"].clip(upper=100) # Some cases in here they report > 100% population affected. 
#                                                                   # We played with this when disagreggating and capping to 100% 
#                                                                   # Since we are using Total Affection again, we need to cap again
#     elif level == "ADM1":
#         group_keys = ["DisNo.", "GID_0", "GID_1"]
#         grouped = (
#             df_predictions.groupby(group_keys)
#             .agg(
#                 prediction_ppl=("predicted", "sum"),
#                 reported=("reported", "max"),
#                 population=("population", "sum"),
#                 total_affected=("Total Affected", "max"),
#                 wind_speed=("wind_speed", "max"),
#             )
#             .reset_index()
#         )

#         # grouped["reported"] = 100 * grouped["actual_ppl"] / grouped["population"]
#         grouped["reported"] = grouped["reported"].clip(upper=100)
#     else:
#         raise ValueError("Level must be 'ADM0' or 'ADM1'.")

#     # Compute predicted percentage
#     grouped["predicted"] = 100 * grouped["prediction_ppl"] / grouped["population"]

#     # Cap predictions at 100
#     grouped["predicted"] = grouped["predicted"].clip(upper=100)

#     # Compute errors
#     grouped["error"] = grouped["predicted"] - grouped["reported"]
#     grouped["abs_error"] = grouped["error"].abs()

#     # Rename total_affected for clarity
#     grouped.rename(columns={"total_affected": "Total Affected"}, inplace=True)

#     return grouped.reset_index(drop=True)


def compute_grouped_predictions(df_test):
    group_keys = ["DisNo.", "GID_0", "GID_1"]
    grouped = (
        df_test.groupby(group_keys)
        .agg(
            prediction_ppl=("predicted", "sum"),
            actual_ppl=("reported_ppl", "sum"),
            actual_perc=("reported", "max"),
            population=("population", "sum")
        )
        .reset_index()
    )
    grouped["reported"] = grouped["actual_perc"] 
    grouped["reported"] = grouped["reported"].clip(upper=100)


    # Compute predicted percentage
    grouped["predicted"] = 100 * grouped["prediction_ppl"] / grouped["population"]

    # Cap predictions at 100
    grouped["predicted"] = grouped["predicted"].clip(upper=100)

    # Compute errors
    grouped["error"] = grouped["predicted"] - grouped["reported"]
    grouped["abs_error"] = grouped["error"].abs()

    return grouped.reset_index(drop=True)


In [6]:
wind_based_model = compute_historical_predictions(df)

In [7]:
# wind_model_adm0 = compute_grouped_predictions(wind_based_model, "ADM0")
# wind_model_adm1 = compute_grouped_predictions(wind_based_model, "ADM1")
wind_model_adm1 = compute_grouped_predictions(wind_based_model)

In [8]:
wind_model_adm1[wind_model_adm1.reported >= 15].shape

(613, 11)

In [9]:
# wind_model_adm0.to_parquet("/data/big/fmoss/data/model_output/merged/adm0_grouped/wind_historical_model.parquet", index=False)
wind_model_adm1.to_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/wind_historical_model.parquet", index=False)

In [ ]:
# Load models
# wind_model_adm0 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm0_grouped/wind_historical_model.parquet")
wind_model_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/wind_historical_model.parquet")


In [15]:
from sklearn.metrics import cohen_kappa_score, f1_score, precision_score, recall_score, confusion_matrix
import numpy as np
import pandas as pd

def safe_quadratic_weighted_kappa(y_true, y_pred):
    unique_true = np.unique(y_true)
    unique_pred = np.unique(y_pred)
    
    if len(unique_true) < 2 or len(unique_pred) < 2:
        # Not enough categories to compute QWK
        return 1
    
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

def evaluate_predictions(df):
    df = df.copy()
    # Compute error metrics
    mae = df["abs_error"].mean()
    std_error = df["abs_error"].std()

    # Define category bins & labels
    bins = [-np.inf, 1, 5, 15, np.inf]
    labels = [0, 1, 2, 3]  # 0: very low, 1: low, 2: moderate, 3: high

    # Categorize reported and predicted
    df["reported_cat"] = pd.cut(df["reported"], bins=bins, labels=labels).astype(int)
    df["predicted_cat"] = pd.cut(df["predicted"], bins=bins, labels=labels).astype(int)

    # Quadratic weighted kappa
    kappa = cohen_kappa_score(df["reported_cat"], df["predicted_cat"], weights="quadratic")
    # kappa = safe_quadratic_weighted_kappa(df["reported_cat"], df["predicted_cat"])
    accuracy = (df["reported_cat"] == df["predicted_cat"]).mean()
    # Binarize: 0,1 → 0 (low); 2,3 → 1 (moderate/high)
    df["reported_bin"] = (df["reported_cat"] >= 2).astype(int)
    df["predicted_bin"] = (df["predicted_cat"] >= 2).astype(int)

    # F1, precision, recall
    f1 = f1_score(df["reported_bin"], df["predicted_bin"], zero_division=np.nan)
    precision = precision_score(df["reported_bin"], df["predicted_bin"], zero_division=np.nan)
    recall = recall_score(df["reported_bin"], df["predicted_bin"], zero_division=np.nan)

    # Helper to round to 2 significant digits
    def sig(x):
        return float(f"{x:.2g}")

    cm = confusion_matrix(df["reported_bin"], df["predicted_bin"], labels=[0,1])
    tn, fp, fn, tp = cm.ravel()

    # Negative precision (NPV)
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    # Negative recall (specificity / TNR)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    # Helper to round to 2 significant digits
    def sig(x):
        return float(f"{x:.3g}")

    # Return metrics rounded to 2 significant digits
    return {
        "mae": sig(mae),
        "std_error": sig(std_error),
        "quadratic_weighted_kappa": sig(kappa),
        "accuracy": sig(accuracy),
        "precision": sig(precision),
        "recall": sig(recall),
        "f1": sig(f1),
        "NPV": sig(npv),
        "specificity": sig(specificity)
    }

In [16]:
metrics_adm0 = evaluate_predictions(wind_model_adm0)
print(metrics_adm0)

{'mae': 10.6, 'std_error': 23.6, 'quadratic_weighted_kappa': 0.238, 'accuracy': 0.554, 'precision': 0.911, 'recall': 0.177, 'f1': 0.296, 'NPV': 0.745, 'specificity': 0.993}


In [17]:
metrics_adm1 = evaluate_predictions(wind_model_adm1.dropna()) # there are 4 adm1 regions with no population
print(metrics_adm1)

{'mae': 0.98, 'std_error': 7.23, 'quadratic_weighted_kappa': 0.134, 'accuracy': 0.931, 'precision': 0.273, 'recall': 0.0381, 'f1': 0.0669, 'NPV': 0.97, 'specificity': 0.997}


## Post processing: top xx events

How does it performs on the top 100 most impacting events?

In [6]:
top_impacting_events = wind_model_adm0.sort_values("reported")[-100:]["DisNo."].unique()

In [7]:
top_events_adm0 = wind_model_adm0[wind_model_adm0["DisNo."].isin(top_impacting_events)]
top_events_adm1 = wind_model_adm1[wind_model_adm1["DisNo."].isin(top_impacting_events)]

In [8]:
metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)
metrics_adm1 = evaluate_predictions(top_events_adm1)
print(metrics_adm1)

{'mae': 61.8, 'std_error': 33.7, 'quadratic_weighted_kappa': 0.0, 'accuracy': 0.19, 'precision': 1.0, 'recall': 0.34, 'f1': 0.507, 'NPV': 0.0, 'specificity': nan}
{'mae': 5.03, 'std_error': 18.9, 'quadratic_weighted_kappa': 0.134, 'accuracy': 0.868, 'precision': 0.154, 'recall': 0.053, 'f1': 0.0789, 'NPV': 0.932, 'specificity': 0.978}


Performance on 100 less impacting events

In [7]:
top_less_impacting_events = wind_model_adm0.sort_values("reported")[:100]["DisNo."].unique()

In [8]:
top_events_adm0 = wind_model_adm0[wind_model_adm0["DisNo."].isin(top_less_impacting_events)]
top_events_adm1 = wind_model_adm1[wind_model_adm1["DisNo."].isin(top_less_impacting_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1.dropna())
print(metrics_adm1)

{'mae': 0.0144, 'std_error': 0.062, 'quadratic_weighted_kappa': nan, 'accuracy': 1.0, 'precision': nan, 'recall': nan, 'f1': nan, 'NPV': 1.0, 'specificity': 1.0}
{'mae': 0.00343, 'std_error': 0.0675, 'quadratic_weighted_kappa': 0.0, 'accuracy': 0.999, 'precision': nan, 'recall': nan, 'f1': nan, 'NPV': 1.0, 'specificity': 1.0}


/home/fmoss/miniforge3/envs/rasterio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fmoss/miniforge3/envs/rasterio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [ ]:
len(top_events_adm0[(top_events_adm0.reported < 5) & (top_events_adm0.predicted < 5)]) # Perfect binaryn modelmodel

How does it performs on the top 100 events with the highest recorded windspeed

In [ ]:
top_windspeed_events = wind_model_adm0.sort_values("wind_speed")[-100:]["DisNo."].unique()

In [ ]:
len(set(top_impacting_events) & set(top_windspeed_events))

In [ ]:
top_windspeed_events_adm0 = wind_model_adm0[wind_model_adm0["DisNo."].isin(top_windspeed_events)]
top_windspeed_events_adm1 = wind_model_adm1[wind_model_adm1["DisNo."].isin(top_windspeed_events)]

In [ ]:
metrics_adm0 = evaluate_predictions(top_windspeed_events_adm0)
print(metrics_adm0)

In [ ]:
metrics_adm1 = evaluate_predictions(top_windspeed_events_adm1.dropna())
print(metrics_adm1)

Rich countries vs Poor countries

In [11]:
gdp = pd.read_excel("/data/big/fmoss/data/GDP/CLASS.xlsx")

emdat_countries = df.iso3.unique()
print(len(emdat_countries))

gdp = gdp[["Code", "Income group"]].rename({"Code": "GID_0"}, axis=1)
gdp = gdp[gdp.GID_0.isin(emdat_countries)]
print(len(gdp))

68
67


In [12]:
wind_model_adm0 = wind_model_adm0.merge(gdp, how="left")
wind_model_adm0["Income group"] = wind_model_adm0["Income group"].fillna("Upper middle income")

wind_model_adm1 = wind_model_adm1.merge(gdp, how="left")
wind_model_adm1["Income group"] = wind_model_adm1["Income group"].fillna("Upper middle income")

rich_countries_events = wind_model_adm0[
    wind_model_adm0["Income group"].isin(["High income", "Upper middle income"])
    ]["DisNo."].unique()

poor_countries_events = wind_model_adm0[
    ~wind_model_adm0["Income group"].isin(["High income", "Upper middle income"])
    ]["DisNo."].unique()


print(len(rich_countries_events))
print(len(poor_countries_events))

408
385


In [13]:
# RICH subset
top_events_adm0 = wind_model_adm0[wind_model_adm0["DisNo."].isin(rich_countries_events)]
top_events_adm1 = wind_model_adm1[wind_model_adm1["DisNo."].isin(rich_countries_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1.dropna())
print(metrics_adm1)

print(len(top_events_adm0.GID_0.unique()))
print(len(rich_countries_events))

{'mae': 8.35, 'std_error': 22.0, 'quadratic_weighted_kappa': 0.287, 'accuracy': 0.662, 'precision': 0.95, 'recall': 0.209, 'f1': 0.342, 'NPV': 0.814, 'specificity': 0.997}
{'mae': 1.47, 'std_error': 10.2, 'quadratic_weighted_kappa': 0.0931, 'accuracy': 0.947, 'precision': 0.16, 'recall': 0.0546, 'f1': 0.0814, 'NPV': 0.976, 'specificity': 0.993}
40
408


In [14]:
# POOR subset
top_events_adm0 = wind_model_adm0[wind_model_adm0["DisNo."].isin(poor_countries_events)]
top_events_adm1 = wind_model_adm1[wind_model_adm1["DisNo."].isin(poor_countries_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1.dropna())
print(metrics_adm1)

print(len(top_events_adm0.GID_0.unique()))
print(len(poor_countries_events))

{'mae': 13.0, 'std_error': 25.0, 'quadratic_weighted_kappa': 0.196, 'accuracy': 0.439, 'precision': 0.88, 'recall': 0.156, 'f1': 0.265, 'NPV': 0.669, 'specificity': 0.988}
{'mae': 0.658, 'std_error': 4.28, 'quadratic_weighted_kappa': 0.165, 'accuracy': 0.921, 'precision': 0.679, 'recall': 0.0304, 'f1': 0.0581, 'NPV': 0.966, 'specificity': 0.999}
28
385
